# Bridge SDE Visualization

This notebook previews the bridge SDE geometry before running training.
It uses shared project utilities from `src/cfm_project` (no duplicated simulation logic).

## 1) Parameters

In [ ]:
from datetime import datetime
from pathlib import Path
import sys

import torch
from matplotlib import pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from cfm_project.bridge_sde import (
    default_bridge_preview_parameters,
    sample_bridge_sde_at_times,
    simulate_bridge_sde_trajectories,
)
from cfm_project.plotting import (
    plot_bridge_snapshot_grid,
    plot_bridge_y_spread,
    save_bridge_animation,
)

params = default_bridge_preview_parameters()
params

## 2) Simulate Trajectories

In [ ]:
seed = 7
generator = torch.Generator(device="cpu")
generator.manual_seed(seed)

times, trajectories = simulate_bridge_sde_trajectories(
    n_samples=int(params["n_samples"]),
    n_steps=int(params["n_steps"]),
    total_time=float(params["total_time"]),
    mean0=params["mean0"],
    cov0=params["cov0"],
    vx=float(params["vx"]),
    sigma_x=float(params["sigma_x"]),
    sigma_y=float(params["sigma_y"]),
    bridge_center_x=float(params["bridge_center_x"]),
    bridge_width=float(params["bridge_width"]),
    bridge_pull=float(params["bridge_pull"]),
    bridge_diffusion_drop=float(params["bridge_diffusion_drop"]),
    generator=generator,
    device="cpu",
    dtype=torch.float32,
)

snapshot_times = [0.00, 0.25, 0.50, 0.75, 1.00]
snapshots = sample_bridge_sde_at_times(snapshot_times, times, trajectories)
times.shape, trajectories.shape

## 3) Static Diagnostics

In [ ]:
fig_grid, _ = plot_bridge_snapshot_grid(snapshots, bins=90, max_points=2200)
plt.show()

fig_spread, _ = plot_bridge_y_spread(times, trajectories)
plt.show()

## 4) Export Figures and Animation

In [ ]:
run_tag = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
preview_dir = ROOT / "outputs" / "preview_bridge_sde" / run_tag
preview_dir.mkdir(parents=True, exist_ok=True)

grid_path = preview_dir / "snapshot_grid.png"
spread_path = preview_dir / "y_spread.png"
anim_path = preview_dir / "bridge_animation.gif"

fig_grid.savefig(grid_path, dpi=140)
fig_spread.savefig(spread_path, dpi=140)
save_bridge_animation(times, trajectories, anim_path, max_points=1200, frame_stride=2, interval_ms=70)

print(f"Saved preview artifacts in: {preview_dir}")

## 5) Quick Summary

In [ ]:
std_y = torch.std(trajectories[:, :, 1], dim=0)
idx_0 = int(torch.argmin(torch.abs(times - 0.0)).item())
idx_05 = int(torch.argmin(torch.abs(times - 0.5)).item())
idx_1 = int(torch.argmin(torch.abs(times - 1.0)).item())
idx_min = int(torch.argmin(std_y).item())

print(f"std(Y_t=0.00): {std_y[idx_0].item():.4f}")
print(f"std(Y_t=0.50): {std_y[idx_05].item():.4f}")
print(f"std(Y_t=1.00): {std_y[idx_1].item():.4f}")
print(f"minimum std(Y_t): {std_y[idx_min].item():.4f} at t={times[idx_min].item():.3f}")